# CIC-IDS2017 — ML Suitability Evaluation

## Objective

Evaluate the practical suitability of CIC-IDS2017 for the machine-learning Intrusion Detection System (IDS) project using the findings established in the preceding analysis notebooks.

This notebook consolidates:

- Dataset scale and structure
- Data-quality concerns
- Class imbalance
- Attack-class representation
- Feature redundancy
- Feature distribution characteristics
- Duplicate-record burden
- Preprocessing requirements
- Computational considerations
- Potential train/test leakage risks

The goal is not to train a final model, but to determine whether CIC-IDS2017 is a suitable candidate for subsequent ML experimentation and IDS development.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../results/cicids2017")

QUALITY_DIR = RESULTS_DIR / "02_feature_data_quality"
CLASS_DIR = RESULTS_DIR / "03_class_distribution"
DIST_DIR = RESULTS_DIR / "04_feature_distribution"
PREP_DIR = RESULTS_DIR / "05_preprocessing"

OUTPUT_DIR = RESULTS_DIR / "06_ml_suitability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Previous Results

In [2]:
feature_quality = pd.read_csv(
    QUALITY_DIR / "feature_quality_summary.csv"
)

high_correlation = pd.read_csv(
    QUALITY_DIR / "high_correlation_pairs.csv"
)

infinite_summary = pd.read_csv(
    QUALITY_DIR / "infinite_value_summary.csv"
)

missing_summary = pd.read_csv(
    QUALITY_DIR / "missing_value_summary.csv"
)

near_constant = pd.read_csv(
    QUALITY_DIR / "near_constant_features.csv"
)

class_distribution = pd.read_csv(
    CLASS_DIR / "overall_class_distribution.csv"
)

class_imbalance = pd.read_csv(
    CLASS_DIR / "class_imbalance_summary.csv"
)

class_presence = pd.read_csv(
    CLASS_DIR / "class_file_presence.csv"
)

feature_distribution = pd.read_csv(
    DIST_DIR / "feature_distribution_summary.csv"
)

outlier_summary = pd.read_csv(
    DIST_DIR / "feature_outlier_summary.csv"
)

preprocessing_decisions = pd.read_csv(
    PREP_DIR / "preprocessing_decisions.csv"
)

preprocessing_summary = pd.read_csv(
    PREP_DIR / "preprocessing_summary.csv"
)

## 3. Establish Dataset Level Metrics

In [3]:
dataset_metrics = {
    "records": 2_830_743,
    "original_features": 78,
    "original_columns": 79,
    "source_files": 8,
    "traffic_classes": 15,
    "benign_percentage": 80.30,
    "missing_values": 1_358,
    "missing_percentage": 0.048,
    "duplicate_occurrences": 308_381,
    "duplicate_group_rows": 403_550,
    "duplicate_groups": 95_150,
    "constant_features": 8,
}

dataset_metrics

{'records': 2830743,
 'original_features': 78,
 'original_columns': 79,
 'source_files': 8,
 'traffic_classes': 15,
 'benign_percentage': 80.3,
 'missing_values': 1358,
 'missing_percentage': 0.048,
 'duplicate_occurrences': 308381,
 'duplicate_group_rows': 403550,
 'duplicate_groups': 95150,
 'constant_features': 8}

## 4. Data Quality Assessment

### Data Quality Assessment

The earlier notebooks identified several data-quality characteristics that affect ML readiness.

The purpose of this section is to determine whether these issues represent blockers or manageable preprocessing requirements.

In [4]:
data_quality_assessment = pd.DataFrame([
    {
        "factor": "Missing values",
        "finding": "1,358 missing values (0.048%)",
        "severity": "Low",
        "ml_impact": "Manageable through imputation"
    },
    {
        "factor": "Infinite values",
        "finding": "Present in rate-based features",
        "severity": "Moderate",
        "ml_impact": "Must be converted/handled before modelling"
    },
    {
        "factor": "Constant features",
        "finding": "8 constant features",
        "severity": "Low",
        "ml_impact": "Can be safely removed"
    },
    {
        "factor": "Duplicate records",
        "finding": "403,550 rows participate in duplicate groups",
        "severity": "High",
        "ml_impact": "Requires explicit handling to reduce leakage risk"
    },
    {
        "factor": "Feature redundancy",
        "finding": "Highly/perfectly correlated feature pairs exist",
        "severity": "Moderate",
        "ml_impact": "Requires feature-selection consideration"
    },
    {
        "factor": "Extreme values",
        "finding": "Heavy-tailed distributions and substantial extremes",
        "severity": "Moderate",
        "ml_impact": "Requires model/transform-aware preprocessing"
    }
])

data_quality_assessment

,factor,finding,severity,ml_impact
0,Missing values,"1,358 missing values (0.048%)",Low,Manageable through imputation
1,Infinite values,Present in rate-based features,Moderate,Must be converted/handled before modelling
2,Constant features,8 constant features,Low,Can be safely removed
3,Duplicate records,"403,550 rows participate in duplicate groups",High,Requires explicit handling to reduce leakage risk
4,Feature redundancy,Highly/perfectly correlated feature pairs exist,Moderate,Requires feature-selection consideration
5,Extreme values,Heavy-tailed distributions and substantial ext...,Moderate,Requires model/transform-aware preprocessing


## 5. Class Distribution Assessment

### Class Distribution Assessment

CIC-IDS2017 contains 15 traffic classes, including BENIGN and 14 attack categories.

The dataset is strongly imbalanced at both the binary and multiclass levels.

This does not make the dataset unsuitable by itself, but it means that accuracy alone would be an inadequate evaluation metric.

In [5]:
class_assessment = pd.DataFrame([
    {
        "factor": "BENIGN dominance",
        "finding": "Approximately 80.30% of all records",
        "severity": "High",
        "ml_impact": "Requires imbalance-aware evaluation"
    },
    {
        "factor": "Rare attack classes",
        "finding": "Several classes contain extremely few observations",
        "severity": "High",
        "ml_impact": "Limits reliable class-specific evaluation"
    },
    {
        "factor": "Attack imbalance",
        "finding": "Attack categories vary substantially in size",
        "severity": "High",
        "ml_impact": "Requires class-aware metrics and modelling strategy"
    },
    {
        "factor": "Class/session concentration",
        "finding": "Some attacks are concentrated in specific source files",
        "severity": "High",
        "ml_impact": "Train/test splitting must consider capture-session composition"
    }
])

class_assessment

,factor,finding,severity,ml_impact
0,BENIGN dominance,Approximately 80.30% of all records,High,Requires imbalance-aware evaluation
1,Rare attack classes,Several classes contain extremely few observat...,High,Limits reliable class-specific evaluation
2,Attack imbalance,Attack categories vary substantially in size,High,Requires class-aware metrics and modelling str...
3,Class/session concentration,Some attacks are concentrated in specific sour...,High,Train/test splitting must consider capture-ses...


## 6. Feature Suitability

In [6]:
feature_assessment = pd.DataFrame([
    {
        "factor": "Feature scale",
        "finding": "Features span substantially different numerical ranges",
        "severity": "Moderate",
        "ml_impact": "Scaling required for scale-sensitive models"
    },
    {
        "factor": "Skewness",
        "finding": "Many features are strongly right-skewed",
        "severity": "Moderate",
        "ml_impact": "May require transformations or robust models"
    },
    {
        "factor": "Feature redundancy",
        "finding": "Highly correlated/perfectly correlated pairs exist",
        "severity": "Moderate",
        "ml_impact": "Feature selection may reduce redundancy"
    },
    {
        "factor": "Constant features",
        "finding": "8 features have no variance",
        "severity": "Low",
        "ml_impact": "Remove during preprocessing"
    },
    {
        "factor": "Extreme observations",
        "finding": "Large ranges and heavy tails occur in multiple features",
        "severity": "Moderate",
        "ml_impact": "Requires careful preprocessing/model selection"
    }
])

feature_assessment

,factor,finding,severity,ml_impact
0,Feature scale,Features span substantially different numerica...,Moderate,Scaling required for scale-sensitive models
1,Skewness,Many features are strongly right-skewed,Moderate,May require transformations or robust models
2,Feature redundancy,Highly correlated/perfectly correlated pairs e...,Moderate,Feature selection may reduce redundancy
3,Constant features,8 features have no variance,Low,Remove during preprocessing
4,Extreme observations,Large ranges and heavy tails occur in multiple...,Moderate,Requires careful preprocessing/model selection


## 7. Computational Suitability

In [7]:
computational_assessment = pd.DataFrame([
    {
        "factor": "Dataset size",
        "finding": "2.83 million records",
        "assessment": "Manageable but computationally significant"
    },
    {
        "factor": "Feature dimensionality",
        "finding": "78 original features",
        "assessment": "Moderate"
    },
    {
        "factor": "Memory requirements",
        "finding": "Full-data operations can require substantial RAM",
        "assessment": "Careful memory management required"
    },
    {
        "factor": "Exploratory analysis",
        "finding": "Sampling was required for some expensive distribution operations",
        "assessment": "Sampling is appropriate for visualization-heavy analysis"
    },
    {
        "factor": "ML experimentation",
        "finding": "Full-dataset experimentation may be expensive",
        "assessment": "Use controlled subsets during initial experimentation"
    }
])

computational_assessment

,factor,finding,assessment
0,Dataset size,2.83 million records,Manageable but computationally significant
1,Feature dimensionality,78 original features,Moderate
2,Memory requirements,Full-data operations can require substantial RAM,Careful memory management required
3,Exploratory analysis,Sampling was required for some expensive distr...,Sampling is appropriate for visualization-heav...
4,ML experimentation,Full-dataset experimentation may be expensive,Use controlled subsets during initial experime...


## 8. Leakage Risk

###Data Leakage Risk

CIC-IDS2017 contains substantial duplicate observations and attack classes that may be concentrated within individual capture sessions.

A naïve random train/test split may therefore produce highly similar observations in both partitions and potentially overestimate model generalization.

The final ML pipeline should therefore explicitly consider:

- Duplicate handling before partitioning
- Capture-session/file composition
- Stratification where appropriate
- Separation of highly similar observations
- Evaluation using metrics beyond overall accuracy

This is a major methodological consideration rather than a reason to reject the dataset outright.

## 9. Overall Suitability Scorecard

In [8]:
suitability_scorecard = pd.DataFrame([
    ["Data volume", "Strong", "Large number of observations available"],
    ["Feature availability", "Strong", "78 original numerical features"],
    ["Attack coverage", "Moderate", "14 attack categories represented"],
    ["Class balance", "Weak", "Strong multiclass and binary imbalance"],
    ["Rare-class representation", "Weak", "Several extremely rare attack classes"],
    ["Data quality", "Moderate", "Issues are present but largely manageable"],
    ["Feature redundancy", "Moderate", "Highly correlated features require review"],
    ["Preprocessing burden", "Moderate", "Multiple explicit preprocessing steps required"],
    ["Computational feasibility", "Moderate", "Large dataset requires memory-aware processing"],
    ["Leakage risk", "Weak", "Duplicates and capture-session concentration require careful splitting"],
    ["Overall ML suitability", "Suitable with conditions", "Viable if preprocessing and evaluation are carefully designed"],
], columns=[
    "criterion",
    "rating",
    "assessment"
])

suitability_scorecard

,criterion,rating,assessment
0,Data volume,Strong,Large number of observations available
1,Feature availability,Strong,78 original numerical features
2,Attack coverage,Moderate,14 attack categories represented
3,Class balance,Weak,Strong multiclass and binary imbalance
4,Rare-class representation,Weak,Several extremely rare attack classes
5,Data quality,Moderate,Issues are present but largely manageable
6,Feature redundancy,Moderate,Highly correlated features require review
7,Preprocessing burden,Moderate,Multiple explicit preprocessing steps required
8,Computational feasibility,Moderate,Large dataset requires memory-aware processing
9,Leakage risk,Weak,Duplicates and capture-session concentration r...


## 10. Final Recommendation

## Final Recommendation

### CIC-IDS2017 is suitable for ML-based IDS development, with methodological constraints.

CIC-IDS2017 provides substantial data volume, broad attack coverage, and a relatively rich feature space, making it a viable dataset for the intended IDS project.

However, it should not be treated as a clean or naturally balanced modelling dataset.

The primary concerns are:

1. Strong class imbalance
2. Extremely rare attack categories
3. Large numbers of duplicate observations
4. Capture-session-specific class representation
5. Feature redundancy
6. Heavy-tailed numerical distributions
7. Significant computational requirements

These concerns are manageable through a carefully designed preprocessing and evaluation pipeline.

In particular, the final modelling workflow should avoid naïve random splitting, explicitly handle duplicate records and invalid numerical values, account for class imbalance, and evaluate models using class-sensitive metrics rather than accuracy alone.

Therefore, CIC-IDS2017 is **not rejected**, but its suitability should ultimately be compared against CIC-IDS2018 and CIC-DDoS2019 before selecting the final dataset for the IDS implementation.

## 11. Export

In [9]:
data_quality_assessment.to_csv(
    OUTPUT_DIR / "data_quality_assessment.csv",
    index=False
)

class_assessment.to_csv(
    OUTPUT_DIR / "class_distribution_assessment.csv",
    index=False
)

feature_assessment.to_csv(
    OUTPUT_DIR / "feature_suitability_assessment.csv",
    index=False
)

computational_assessment.to_csv(
    OUTPUT_DIR / "computational_feasibility_assessment.csv",
    index=False
)

suitability_scorecard.to_csv(
    OUTPUT_DIR / "ml_suitability_scorecard.csv",
    index=False
)

pd.DataFrame([dataset_metrics]).to_csv(
    OUTPUT_DIR / "dataset_metrics.csv",
    index=False
)

print("ML suitability assessment exported successfully.")

ML suitability assessment exported successfully.


## 12. Final Summary

## Conclusion

CIC-IDS2017 is a viable candidate for machine-learning IDS development due to its large scale, broad attack coverage, and substantial feature set.

The analysis also identifies important limitations. Strong class imbalance, rare attack classes, duplicate records, capture-session concentration, feature redundancy, heavy-tailed distributions, and computational requirements can affect both model training and the reliability of evaluation.

These limitations do not make the dataset unsuitable, but they require an explicit preprocessing and evaluation strategy.

At this stage, CIC-IDS2017 is classified as **suitable with conditions**. A final dataset selection will be made only after applying the same evaluation framework to CIC-IDS2018 and CIC-DDoS2019 and comparing the resulting suitability profiles.